In [ ]:
from google.colab import drive
MAIN_FOLDER="/content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data"
DATA_FOLDER=MAIN_FOLDER+"/data"
CODE_FOLDER=MAIN_FOLDER+"/code"
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import sys
# Append the folder path to sys.path so Python can find the local modules
if MAIN_FOLDER not in sys.path:
    sys.path.append(MAIN_FOLDER)
    print(f'Added {MAIN_FOLDER} to sys.path')
if DATA_FOLDER not in sys.path:
    sys.path.append(DATA_FOLDER)
    print(f'Added {DATA_FOLDER} to sys.path')
if CODE_FOLDER not in sys.path:
    sys.path.append(CODE_FOLDER)
    print(f'Added {CODE_FOLDER} to sys.path')

Added /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data to sys.path
Added /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data/data to sys.path
Added /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data/code to sys.path


In [ ]:
from __future__ import annotations

import torch
from visualize import extract_and_plot
import lm
import torch
from torch import nn, optim
from transformer import TransformerLM
import data
!pip install optuna

import optuna
import torch
import torch.optim as optim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 9.1 MB/s eta 0:00:00


In [ ]:
model: torch.nn.Module = TransformerLM(
        n_layers,
        n_heads,
        embed_size,
        seq_len,
        tokenizer.vocab_size(),
        mlp_hidden_size,
        with_residuals=True,
    )

model = model.to(device)
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, betas=[0.9, 0.95])
num_batches = 0

Parameter count: 2.51M


In [ ]:
import os

# Define the paths for the directories
models_dir = os.path.join(MAIN_FOLDER, 'models') # Use MAIN_FOLDER for relative path consistency
figs_dir = os.path.join(MAIN_FOLDER, 'figs')

# Check if the 'models' directory exists, and create it if not
if not os.path.exists(models_dir):
    os.makedirs(models_dir)
    print(f"Created directory: {models_dir}")
else:
    print(f"Directory already exists: {models_dir}")

# Check if the 'figs' directory exists, and create it if not
if not os.path.exists(figs_dir):
    os.makedirs(figs_dir)
    print(f"Created directory: {figs_dir}")
else:
    print(f"Directory already exists: {figs_dir}")

Directory already exists: /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data/models
Directory already exists: /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data/figs


In [ ]:

from utils import save_best_model,parameters,loss_plotter,split_data
def objective(trial):
    # 1. Define the hyperparameter search space
    seq_len = trial.suggest_categorical("seq_len", [128, 256])
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
    n_layers = trial.suggest_categorical("n_layers", [4, 6, 8])
    n_heads = trial.suggest_categorical("n_heads", [4,8])
    embed_size = trial.suggest_categorical("embed_size", [128, 256, 384])
    dropouts=trial.suggest_categorical("dropouts", [0.1, 0.2, 0.3])

    # Assuming learning rate should be a log-uniform float. Adjust bounds as needed.
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-2, log=True)

    mlp_hidden_size = embed_size * 4

    params = {
        "seq_len": seq_len,
        "batch_size": batch_size,
        "n_layers": n_layers,
        "n_heads": n_heads,
        "embed_size": embed_size,
        "mlp_hidden_size": mlp_hidden_size,
        "learning_rate": learning_rate,
        "dropout_rate": dropouts
    }

    # 2. Data Split and Iterators
    train_data, val_data = split_data(tokenized_data, seq_len)

    train_iter = iter(data.RandomOrderDataIterator(train_data, seq_len + 1))
    val_iter = iter(data.RandomOrderDataIterator(val_data, seq_len + 1))

    print(f'Training model with params: {params}')
    # 3. Initialize Model
    model: torch.nn.Module = TransformerLM(
        params['n_layers'],
        params['n_heads'],
        params['embed_size'],
        params['seq_len'],
        tokenizer.vocab_size(),
        params['mlp_hidden_size'],
        with_residuals=True,
        dropout=[params["dropout_rate"],params["dropout_rate"],params["dropout_rate"]]
    )
    model = model.to(device)

    # 4. Setup Optimizer & Scheduler
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, betas=[0.9, 0.95])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=5,
        threshold=1e-4,
        min_lr=1e-7
    )

    # 5. Training Loop Variables
    best_val_loss = float('inf')
    val_losses = []
    train_losses = []
    num_batches = 0
    early_stopping_patience = 7 # Number of validation checks to wait before stopping
    epochs_no_improve = 0
    model.train()

    for batch in data.batch_items(train_iter, batch_size):
        if num_batches >= num_batches_to_train:
            break

        batch_x, batch_y = lm.batch_to_labeled_samples(batch)
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        logits = model(batch_x)
        loss = lm.compute_loss(logits, batch_y)

        # Parameter update
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), gradient_clipping)
        optimizer.step()

        num_batches += 1

        if num_batches % 10 == 0:
            # print(f"Seen {num_batches} batches. last loss is: {loss.item()}")
            train_losses.append(loss.item())

            # if num_batches % 100 == 0:
                # for _ in range(1):
                    # model.eval()
                    # sampled = tokenizer.detokenize(
                        # model.sample_continuation(tokenizer.tokenize("Hello"), 500)
                    # )
                    # model.train()
                    # print(f"Model sample: '''{sampled}'''")
                # print("")

        if num_batches % 100 == 0:
            model.eval()
            with torch.no_grad():
                val_batch = None
                try:
                    val_batch = next(data.batch_items(val_iter, batch_size))
                except StopIteration:
                    val_iter = iter(data.RandomOrderDataIterator(val_data, seq_len + 1))
                    val_batch = next(data.batch_items(val_iter, batch_size))

                val_x, val_y = lm.batch_to_labeled_samples(val_batch)
                val_x = val_x.to(device)
                val_y = val_y.to(device)

                val_logits = model(val_x)
                val_loss = lm.compute_loss(val_logits, val_y)
                curr_val_loss = val_loss.item()
                val_losses.append(curr_val_loss)
                scheduler.step(curr_val_loss)

                if curr_val_loss < best_val_loss:
                    best_val_loss = curr_val_loss
                    epochs_no_improve = 0  # Reset patience counter
                    # save_best_model(model, params, best_val_loss, epoch=num_batches, out_dir=models_dir)
                else:
                    epochs_no_improve += 1 # Increment patience counter
                # print(f"Val loss: {curr_val_loss} best val loss: {best_val_loss}")

            # --- OPTUNA PRUNING CHECK ---
            trial.report(curr_val_loss, num_batches)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()
            # ---  TRADITIONAL EARLY STOPPING CHECK ---
            if epochs_no_improve >= early_stopping_patience:
                print("Early stopping triggered: Validation loss stagnated.")
                break # Breaks out of the training loop, ending this trial early
            model.train()
        if num_batches %500 ==0:
          print(f"batch: {num_batches} Val loss: {curr_val_loss} best val loss: {best_val_loss}")


    # Plot metrics at the end of the trial
    loss_plotter(train_losses, val_losses, params, figs_dir)

    # 6. Return the objective value for Optuna to minimize
    return best_val_loss

In [ ]:
print(torch.cuda.is_available())
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
data_path = DATA_FOLDER+"/he/"
gradient_clipping = 1.0
num_batches_to_train = 6000
tokenizer, tokenized_data = data.load_data(data_path)
# NOTE: are data items are longer by one than the sequence length,
# They will be shortened by 1 when converted to training examples.
# data_iter = iter(data.RandomOrderDataIterator(tokenized_data, seq_len + 1))

True
Using device: cuda:0


In [ ]:
# Create study and optionally use a pruner (MedianPruner is standard)
study = optuna.create_study(
    direction="minimize",
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=500)
)

# Run for a specified number of trials
study.optimize(objective, n_trials=30)

# Output the best results
print("\nOptimization Finished!")
print("Best trial:")
best_trial = study.best_trial

print(f"  Value (Best Val Loss): {best_trial.value}")
print("  Best Params: ")
for key, value in best_trial.params.items():
    print(f"    {key}: {value}")

[I 2026-04-19 07:53:38,221] A new study created in memory with name: no-name-c84efa15-a4b8-4e5e-b1b5-95da9882d0ed


Training model with params: {'seq_len': 256, 'batch_size': 32, 'n_layers': 6, 'n_heads': 4, 'embed_size': 256, 'mlp_hidden_size': 1024, 'learning_rate': 0.00015740852787473329, 'dropout_rate': 0.2}
Parameter count: 4.46M
batch: 500 Val loss: 2.4758427143096924 best val loss: 2.4758427143096924
batch: 1000 Val loss: 2.2321114540100098 best val loss: 2.2321114540100098
batch: 1500 Val loss: 2.167283296585083 best val loss: 2.167283296585083
batch: 2000 Val loss: 2.1535730361938477 best val loss: 2.113833427429199
batch: 2500 Val loss: 2.1679024696350098 best val loss: 1.9677067995071411
Early stopping triggered: Validation loss stagnated.


[I 2026-04-19 07:59:39,338] Trial 0 finished with value: 1.9677067995071411 and parameters: {'seq_len': 256, 'batch_size': 32, 'n_layers': 6, 'n_heads': 4, 'embed_size': 256, 'dropouts': 0.2, 'learning_rate': 0.00015740852787473329}. Best is trial 0 with value: 1.9677067995071411.


Saved plot to /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data/figs/loss_plot_seq256_bs32_L6_H4_E256_M1024_lr0p0001574085279_2026-04-19_07-59-38.png
Training model with params: {'seq_len': 256, 'batch_size': 128, 'n_layers': 4, 'n_heads': 4, 'embed_size': 128, 'mlp_hidden_size': 512, 'learning_rate': 0.005930295626683788, 'dropout_rate': 0.2}
Parameter count: 0.78M
batch: 500 Val loss: 2.255657434463501 best val loss: 2.255657434463501
batch: 1000 Val loss: 2.113203525543213 best val loss: 2.103473663330078
batch: 1500 Val loss: 2.188781499862671 best val loss: 2.103473663330078
Early stopping triggered: Validation loss stagnated.


[I 2026-04-19 08:04:04,875] Trial 1 finished with value: 2.103473663330078 and parameters: {'seq_len': 256, 'batch_size': 128, 'n_layers': 4, 'n_heads': 4, 'embed_size': 128, 'dropouts': 0.2, 'learning_rate': 0.005930295626683788}. Best is trial 0 with value: 1.9677067995071411.


...

Training model with params: {'seq_len': 256, 'batch_size': 128, 'n_layers': 8, 'n_heads': 4, 'embed_size': 384, 'mlp_hidden_size': 1536, 'learning_rate': 9.082624010446874e-05, 'dropout_rate': 0.1}
Parameter count: 13.18M


[I 2026-04-19 09:44:41,056] Trial 28 pruned. 


Training model with params: {'seq_len': 256, 'batch_size': 32, 'n_layers': 6, 'n_heads': 4, 'embed_size': 128, 'mlp_hidden_size': 512, 'learning_rate': 0.0012029837980029772, 'dropout_rate': 0.2}
Parameter count: 1.15M


[I 2026-04-19 09:45:12,650] Trial 29 pruned. 



Optimization Finished!
Best trial:
  Value (Best Val Loss): 1.954553484916687
  Best Params: 
    seq_len: 128
    batch_size: 64
    n_layers: 6
    n_heads: 4
    embed_size: 256
    dropouts: 0.3
    learning_rate: 0.0005362054965802242


In [ ]:
print(torch.cuda.is_available())
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
data_path = DATA_FOLDER+"/en/"
gradient_clipping = 1.0
num_batches_to_train = 6000
tokenizer, tokenized_data = data.load_data(data_path)
# NOTE: are data items are longer by one than the sequence length,
# They will be shortened by 1 when converted to training examples.
# data_iter = iter(data.RandomOrderDataIterator(tokenized_data, seq_len + 1))

True
Using device: cuda:0


In [ ]:
# Create study and optionally use a pruner (MedianPruner is standard)
study = optuna.create_study(
    direction="minimize",
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=500)
)

# Run for a specified number of trials
study.optimize(objective, n_trials=30)

# Output the best results
print("\nOptimization Finished!")
print("Best trial:")
best_trial = study.best_trial

print(f"  Value (Best Val Loss): {best_trial.value}")
print("  Best Params: ")
for key, value in best_trial.params.items():
    print(f"    {key}: {value}")

[I 2026-04-19 10:13:51,246] A new study created in memory with name: no-name-0185460e-d61a-43d0-bc87-830d770e25a3


Training model with params: {'seq_len': 128, 'batch_size': 128, 'n_layers': 4, 'n_heads': 8, 'embed_size': 256, 'mlp_hidden_size': 1024, 'learning_rate': 8.442106139757634e-05, 'dropout_rate': 0.1}
Parameter count: 2.96M
batch: 500 Val loss: 2.2844462394714355 best val loss: 2.2844462394714355
batch: 1000 Val loss: 2.048492431640625 best val loss: 2.048492431640625
batch: 1500 Val loss: 1.926980972290039 best val loss: 1.926980972290039
batch: 2000 Val loss: 1.8322646617889404 best val loss: 1.8322646617889404
batch: 2500 Val loss: 1.7789394855499268 best val loss: 1.7664076089859009
batch: 3000 Val loss: 1.6993274688720703 best val loss: 1.6993274688720703
batch: 3500 Val loss: 1.6312689781188965 best val loss: 1.6312689781188965
batch: 4000 Val loss: 1.6173845529556274 best val loss: 1.6173845529556274
batch: 4500 Val loss: 1.6176775693893433 best val loss: 1.578757643699646
batch: 5000 Val loss: 1.5895304679870605 best val loss: 1.5720529556274414
batch: 5500 Val loss: 1.58779942989

[I 2026-04-19 10:31:05,121] Trial 0 finished with value: 1.5335807800292969 and parameters: {'seq_len': 128, 'batch_size': 128, 'n_layers': 4, 'n_heads': 8, 'embed_size': 256, 'dropouts': 0.1, 'learning_rate': 8.442106139757634e-05}. Best is trial 0 with value: 1.5335807800292969.


Saved plot to /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data/figs/loss_plot_seq128_bs128_L4_H8_E256_M1024_lr8p44210614em05_2026-04-19_10-31-04.png
Training model with params: {'seq_len': 256, 'batch_size': 32, 'n_layers': 8, 'n_heads': 8, 'embed_size': 128, 'mlp_hidden_size': 512, 'learning_rate': 0.001215249230480346, 'dropout_rate': 0.1}
Parameter count: 1.50M
batch: 500 Val loss: 2.239459276199341 best val loss: 2.239459276199341
batch: 1000 Val loss: 1.911772608757019 best val loss: 1.8921139240264893
batch: 1500 Val loss: 1.735715389251709 best val loss: 1.6879603862762451
batch: 2000 Val loss: 1.6280367374420166 best val loss: 1.5670453310012817
batch: 2500 Val loss: 1.6140573024749756 best val loss: 1.5244277715682983
Early stopping triggered: Validation loss stagnated.


[I 2026-04-19 10:37:33,572] Trial 1 finished with value: 1.5244277715682983 and parameters: {'seq_len': 256, 'batch_size': 32, 'n_layers': 8, 'n_heads': 8, 'embed_size': 128, 'dropouts': 0.1, 'learning_rate': 0.001215249230480346}. Best is trial 1 with value: 1.5244277715682983.


...

Saved plot to /content/drive/MyDrive/לימודים תואר שני/Year_2/semester_2/LLmCourse/ex1/code-and-data/figs/loss_plot_seq128_bs128_L4_H4_E384_M1536_lr0p002056476874_2026-04-19_14-47-12.png
Training model with params: {'seq_len': 128, 'batch_size': 128, 'n_layers': 4, 'n_heads': 4, 'embed_size': 128, 'mlp_hidden_size': 512, 'learning_rate': 0.004070422496396908, 'dropout_rate': 0.1}
Parameter count: 0.76M


[I 2026-04-19 14:47:46,970] Trial 28 pruned. 


Training model with params: {'seq_len': 128, 'batch_size': 128, 'n_layers': 4, 'n_heads': 4, 'embed_size': 384, 'mlp_hidden_size': 1536, 'learning_rate': 9.768844628337484e-05, 'dropout_rate': 0.3}
Parameter count: 6.61M


[I 2026-04-19 14:50:12,975] Trial 29 pruned. 



Optimization Finished!
Best trial:
  Value (Best Val Loss): 1.4172929525375366
  Best Params: 
    seq_len: 128
    batch_size: 128
    n_layers: 4
    n_heads: 4
    embed_size: 384
    dropouts: 0.2
    learning_rate: 0.002124549291981849


In [ ]:
# Retrieve best parameters from the Optuna study
best_params = study.best_trial.params

# Assign best parameters to individual variables
seq_len = best_params['seq_len']
batch_size = best_params['batch_size']
n_layers = best_params['n_layers']
n_heads = best_params['n_heads']
embed_size = best_params['embed_size']
dropout_rate = best_params['dropouts']
learning_rate = best_params['learning_rate']
n_layers = 6
seq_len: 128
batch_size: 64
n_layers: 6
n_heads: 4
embed_size: 256
dropout_rate: 0.3
learning_rate: 0.0005362054965802242
mlp_hidden_size = embed_size * 4
gradient_clipping = 1.0

# Calculate mlp_hidden_size based on embed_size
mlp_hidden_size = embed_size * 4

print(f"Best Parameters: {best_params}")

# Re-initialize tokenizer and tokenized_data for English (as per previous run)
data_path = DATA_FOLDER+"/hb/"
tokenizer, tokenized_data = data.load_data(data_path)

# Data Split and Iterators for the final training
train_data, val_data = split_data(tokenized_data, seq_len)
train_iter = iter(data.RandomOrderDataIterator(train_data, seq_len + 1))
val_iter = iter(data.RandomOrderDataIterator(val_data, seq_len + 1))

# Initialize Model with best parameters
model: torch.nn.Module = TransformerLM(
    n_layers,
    n_heads,
    embed_size,
    seq_len,
    tokenizer.vocab_size(),
    mlp_hidden_size,
    with_residuals=True,
    dropout=[dropout_rate, dropout_rate, dropout_rate]
)
model = model.to(device)
print(f"Parameter count: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f}M")

# Setup Optimizer & Scheduler
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, betas=[0.9, 0.95])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=5,
    threshold=1e-4,
    min_lr=1e-7
)

# Training Loop Variables
best_val_loss = float('inf')
val_losses = []
train_losses = []
num_batches = 0
early_stopping_patience = 10 # Number of validation checks to wait before stopping
epochs_no_improve = 0
model.train()

# Define the total number of batches for the final training
num_batches_to_train = 50000 # Increased batches for final training

print("Starting final training...")

while True:
    for batch in data.batch_items(train_iter, batch_size):
        if num_batches >= num_batches_to_train:
            break

        batch_x, batch_y = lm.batch_to_labeled_samples(batch)
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        logits = model(batch_x)
        loss = lm.compute_loss(logits, batch_y)

        # Parameter update
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), gradient_clipping)
        optimizer.step()

        num_batches += 1
        train_losses.append(loss.item())

        if num_batches % 100 == 0:
            print(f"Seen {num_batches} batches. last train loss: {loss.item():.4f}")

            model.eval()
            with torch.no_grad():
                val_batch = None
                try:
                    val_batch = next(data.batch_items(val_iter, batch_size))
                except StopIteration:
                    val_iter = iter(data.RandomOrderDataIterator(val_data, seq_len + 1))
                    val_batch = next(data.batch_items(val_iter, batch_size))

                val_x, val_y = lm.batch_to_labeled_samples(val_batch)
                val_x = val_x.to(device)
                val_y = val_y.to(device)

                val_logits = model(val_x)
                val_loss = lm.compute_loss(val_logits, val_y)
                curr_val_loss = val_loss.item()
                val_losses.append(curr_val_loss)
                scheduler.step(curr_val_loss)

                if curr_val_loss < best_val_loss:
                    best_val_loss = curr_val_loss
                    epochs_no_improve = 0  # Reset patience counter
                    save_best_model(model, best_params, best_val_loss, epoch=num_batches, out_dir=models_dir)
                    print(f"New best validation loss: {best_val_loss:.4f}. Model saved.")
                else:
                    epochs_no_improve += 1 # Increment patience counter

                print(f"Batch: {num_batches}, Val loss: {curr_val_loss:.4f}, Best val loss: {best_val_loss:.4f}")
                sampled = tokenizer.detokenize(
                            model.sample_continuation(tokenizer.tokenize("שלום"), 500)
                        )
                print("sampled:",sampled)
                # Generate attention plot every 100 batches
                test_text = "שלום"
                extract_and_plot(
                    model,
                    tokenizer,
                    prefix_text=test_text,
                    save_path=os.path.join(figs_dir, f"attention_map_batch_{num_batches}.png"),
                    max_len=seq_len
                )
                print(f"Attention map saved for batch {num_batches}")

            # --- EARLY STOPPING CHECK ---
            if epochs_no_improve >= early_stopping_patience:
                print("Early stopping triggered: Validation loss stagnated.")
                break # Breaks out of the inner for loop
            model.train()

    if num_batches >= num_batches_to_train or epochs_no_improve >= early_stopping_patience:
        break # Breaks out of the outer while loop

print("Final training complete.")
# Plot metrics at the end of the final training
loss_plotter(train_losses, val_losses, best_params, figs_dir)